# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PiyushVarman/FRML1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
print("""
Data Contract Definition:
- Unit of Analysis: One row represents one unique webpage URL (content_id) evaluated over a fixed historical aggregation window.
- Time Window: 90-day rolling telemetry window (e.g., assessing mature content where content_age_days >= 90).
- Data Source / Table: Anonymized content refresh telemetry warehouse dataset (simulating warehouse parquet partitions filtered on mid-panel month periods like month = '2026-03').
""")


Data Contract Definition:
- Unit of Analysis: One row represents one unique webpage URL (content_id) evaluated over a fixed historical aggregation window.
- Time Window: 90-day rolling telemetry window (e.g., assessing mature content where content_age_days >= 90).
- Data Source / Table: Anonymized content refresh telemetry warehouse dataset (simulating warehouse parquet partitions filtered on mid-panel month periods like month = '2026-03').



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
print("""
Field Categorization:
1. Features (Inputs):
   - content_age_days (Age of the webpage in days)
   - days_since_last_update (Recency metric since last content modification)
   - impressions_90d (Search visibility volume over the past 90 days)
   - ctr (Click-Through Rate over the 90 days)
   - avg_position (Average search result rank position)

2. Label / Target (Proxy):
   - trend_direction / trend_pct (Observed directional traffic change over the period used to construct the priority score proxy)

3. Context (Metadata / Identifiers):
   - content_id (Unique identifier for the row/URL)
   - client_id (Client/domain grouping identifier)
   - content_type / main_intent (Qualitative classification of the page)

4. Excluded Fields & Rationale:
   - future_impressions_next_30d (Excluded to prevent target leakage; represents future outcome data not knowable at the decision moment).
   - client_revenue_attributable (Excluded due to privacy and client-specific noise that does not generalize across search visibility mechanics).
""")


Field Categorization:
1. Features (Inputs):
   - content_age_days (Age of the webpage in days)
   - days_since_last_update (Recency metric since last content modification)
   - impressions_90d (Search visibility volume over the past 90 days)
   - ctr (Click-Through Rate over the 90 days)
   - avg_position (Average search result rank position)

2. Label / Target (Proxy):
   - trend_direction / trend_pct (Observed directional traffic change over the period used to construct the priority score proxy)

3. Context (Metadata / Identifiers):
   - content_id (Unique identifier for the row/URL)
   - client_id (Client/domain grouping identifier)
   - content_type / main_intent (Qualitative classification of the page)

4. Excluded Fields & Rationale:
   - future_impressions_next_30d (Excluded to prevent target leakage; represents future outcome data not knowable at the decision moment).
   - client_revenue_attributable (Excluded due to privacy and client-specific noise that does not generalize a

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
import pandas as pd

# Load dataset (using our local anonymized content refresh dataset as the warehouse extract)
df = pd.read_csv("https://raw.githubusercontent.com/PiyushVarman/FRML1/main/data/raw/content_refresh_anonymized.csv")

print("=== VERIFICATION QUERY 1: Grain Check (Unique content_id count vs Total rows) ===")
total_rows = len(df)
unique_urls = df["content_id"].nunique()
print(f"Total Rows: {total_rows:,}")
print(f"Unique content_id count: {unique_urls:,}")
assert total_rows == unique_urls, "Grain violation: content_id is not uniquely identifying rows!"
print("-> Grain verified: One row equals exactly one unique webpage URL.\n")

print("=== VERIFICATION QUERY 2: Row Count and Date Span / Maturity Filter ===")
# Filter for mature content (age >= 90 days) with active impressions
df_mature = df[(df["content_age_days"] >= 90) & (df["impressions_90d"] > 0)].copy()
print(f"Total Mature Rows Analyzed: {len(df_mature):,}")
print(f"Content Age Span: min = {df_mature['content_age_days'].min()} days, max = {df_mature['content_age_days'].max()} days\n")

print("=== VERIFICATION QUERY 3: Availability Check with IS TRUE / Non-Null Validations ===")
# Check availability of key feature columns using boolean conditions (IS TRUE equivalent)
valid_mask = (df_mature["impressions_90d"] >= 0) & (df_mature["ctr"].notnull()) & (df_mature["avg_position"].notnull())
surviving_rows = df_mature[valid_mask]
print(f"Rows satisfying data availability filters: {len(surviving_rows):,} out of {len(df_mature):,} mature rows")

# Build our First 5 Features with 'Available when?' declaration
print("\n=== THE FIRST 5 FEATURES & DECISION-MOMENT AVAILABILITY ===")
features_df = surviving_rows[[
    "content_id", "content_age_days", "days_since_last_update", "impressions_90d", "ctr", "avg_position"
]].head(5).copy()

feature_line_notes = [
    "1. content_age_days: Knowable at decision moment because static creation timestamps are permanently recorded.",
    "2. days_since_last_update: Knowable at decision moment because CMS modification logs record the last edit date.",
    "3. impressions_90d: Knowable at decision moment from closed past 90-day Search Console telemetry logs.",
    "4. ctr: Knowable at decision moment from past 90-day clicks divided by past impressions.",
    "5. avg_position: Knowable at decision moment from historical search ranking aggregation logs."
]
for note in feature_line_notes:
    print(note)
print("\nFeature Frame Preview:")
display(features_df)

=== VERIFICATION QUERY 1: Grain Check (Unique content_id count vs Total rows) ===
Total Rows: 30,000
Unique content_id count: 30,000
-> Grain verified: One row equals exactly one unique webpage URL.

=== VERIFICATION QUERY 2: Row Count and Date Span / Maturity Filter ===
Total Mature Rows Analyzed: 30,000
Content Age Span: min = 90 days, max = 564 days

=== VERIFICATION QUERY 3: Availability Check with IS TRUE / Non-Null Validations ===
Rows satisfying data availability filters: 30,000 out of 30,000 mature rows

=== THE FIRST 5 FEATURES & DECISION-MOMENT AVAILABILITY ===
1. content_age_days: Knowable at decision moment because static creation timestamps are permanently recorded.
2. days_since_last_update: Knowable at decision moment because CMS modification logs record the last edit date.
3. impressions_90d: Knowable at decision moment from closed past 90-day Search Console telemetry logs.
4. ctr: Knowable at decision moment from past 90-day clicks divided by past impressions.
5. avg_p

,content_id,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position
0,content_304f48230142,187,20,3803,0.76,10.6
1,content_a1fb4e703a9e,445,25,15320,0.05,20.3
2,content_9aa793d4d895,141,20,12581,0.09,36.5
3,content_331d6c4de07b,463,22,11751,0.49,6.2
4,content_d99b7a2d90ca,263,14,19140,0.13,44.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [6]:
import numpy as np

print("=== THE LEAKAGE EXPERIMENT (Notebook 02 Lesson) ===")
# Add a deliberate label-derived leakage column (e.g., future/contemporaneous target trend percentage or direct post-outcome status)
# In a real model, including a feature directly derived from the target makes performance jump artificially to near 100%.
df_leak = features_df.copy()

# Injecting leakage: adding 'trend_pct' directly from the dataset which mirrors the outcome label
df_leak["LEAK_future_trend_pct"] = df.loc[df_leak.index, "trend_pct"]

print("Feature frame WITH forbidden leakage column:")
display(df_leak.head(3))
print("\nLesson Learned: Including label-derived columns causes metric scores to jump toward perfection due to data leakage, "
      "violating the decision-moment rule. Removing the leakage column restores honest, out-of-sample predictive validity.")

# Cleanly remove the leaked column
df_clean = df_leak.drop(columns=["LEAK_future_trend_pct"])
print("Leakage column removed successfully. Retaining honest feature set.")

=== THE LEAKAGE EXPERIMENT (Notebook 02 Lesson) ===
Feature frame WITH forbidden leakage column:


,content_id,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position,LEAK_future_trend_pct
0,content_304f48230142,187,20,3803,0.76,10.6,-41.4
1,content_a1fb4e703a9e,445,25,15320,0.05,20.3,-57.7
2,content_9aa793d4d895,141,20,12581,0.09,36.5,-60.9



Lesson Learned: Including label-derived columns causes metric scores to jump toward perfection due to data leakage, violating the decision-moment rule. Removing the leakage column restores honest, out-of-sample predictive validity.
Leakage column removed successfully. Retaining honest feature set.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.